# 홈트레이닝 자세 채점 — 시연 노트북

영상 하나를 넣으면: **좌표 추출 → 원본 위 스켈레톤 오버레이 → 전처리(DTW) → 채점
(통계 + 규칙 기반) → 이상치 언어화 피드백**까지 전체 파이프라인이 이어지는 걸
한 번에 보여줍니다.

**미리 준비해두세요** (무거운 단계라 발표 중 라이브로 돌리지 않는 걸 권장합니다):
- `models/{exercise}_reference.npz` — `scoring_model.train`으로 미리 학습
- `data/processed/templates.npz` — `preprocessing.build_dataset`으로 미리 생성
- 시연용 영상 3개 (아래 `DEMO_VIDEOS`에 경로만 채우면 됩니다):
    1. **좋은 폼** — 통계 점수로 이미 높게 나오는 예시
    2. **명확히 나쁜 폼** — 이상치가 뚜렷하게 잡히는 예시
    3. **최소 동작은 통과했지만 통계적으로 크게 벗어난 폼** — mahalanobis 점수가
       (이유를 불문하고) 극단적으로 낮게 나오지만, 규칙 기반 최저점 보장으로
       최대 50점까지는 확보되는 예시

> **주의**: 규칙 기반 보정은 "잘하면 더 준다"가 아니라 **"최소한의 동작 기준(예:
> 일정 깊이 이상 굽힘)만 충족하면, 통계 점수가 어떻게 나오든 최대 50점까지는
> 보장해준다"는 안전망**입니다. 실제로 확인한 깊은 스쿼트 사례는 mahalanobis
> 만으로도 이미 90점대가 나왔고 규칙이 개입하지 않았습니다 — 3번 예시는 규칙의
> 최저점 보장 효과를 명확히 보여주기 위해 통계 점수가 실제로 낮게 나오는 케이스로
> 준비해주세요.

## 0. 환경 설정

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, Image, display

project_root = Path(".").resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.overlay_visualizer import overlay_skeleton_on_video
from src.scoring_model.score_reps import score_video, load_template

plt.rcParams["figure.figsize"] = (9, 4)

# 한글 폰트가 없으면 그래프의 한글 라벨이 네모(□)로 깨질 수 있어, 있는 폰트 중 하나로 자동 설정
import matplotlib.font_manager as fm
_korean_fonts = [f.name for f in fm.fontManager.ttflist if any(
    kw in f.name for kw in ["Malgun", "NanumGothic", "AppleGothic", "Noto Sans CJK", "Noto Sans KR"]
)]
if _korean_fonts:
    plt.rcParams["font.family"] = _korean_fonts[0]
plt.rcParams["axes.unicode_minus"] = False
np.set_printoptions(precision=3, suppress=True)

## 1. 시연 영상 설정

`video_path`(원본 mp4)와 `keypoints_path`(원본 keypoints .npy, 정규화 전)를 짝지어 넣으세요.
`tests/` 폴더처럼 `data/raw`와 분리된 시연용 폴더를 쓰시면 됩니다.

In [ ]:
EXERCISE = "squat"
MODEL_DIR = project_root / "models"
PROCESSED_DIR = project_root / "data" / "processed"

DEMO_VIDEOS = {
    "좋은 폼": {
        "video_path": "tests/demo_good.mp4",
        "keypoints_path": "tests/demo_good_keypoints.npy",
    },
    "나쁜 폼": {
        "video_path": "tests/demo_bad.mp4",
        "keypoints_path": "tests/demo_bad_keypoints.npy",
    },
    "통계상 큰 이상치지만 최소 동작은 충족 (규칙 최저점 보장 시연)": {
        "video_path": "tests/demo_outlier.mp4",
        "keypoints_path": "tests/demo_outlier_keypoints.npy",
    },
}

template = load_template(PROCESSED_DIR, EXERCISE)
print(f"'{EXERCISE}' 템플릿 로드 완료 (길이 {len(template)})")
print(f"시연 영상 {len(DEMO_VIDEOS)}개 등록됨: {list(DEMO_VIDEOS.keys())}")

## 2. 원본 영상 위에 스켈레톤 오버레이

"지금 이 화면에 보이는 영상이 실제로 처리되고 있다"는 걸 직관적으로 보여주는 단계입니다.
파일명이나 경로가 아니라, 원본 픽셀 위에 실시간으로 관절점이 따라다니는 걸 그대로
보여줍니다.

In [ ]:
# "mp4"(기본, 노트북 안 재생용) 또는 "gif"(README/슬라이드 등 외부 공유용 — GitHub
# README는 mp4를 인라인 재생 못 하지만 gif는 자동 재생된다). gif는 색상 제한과
# 큰 파일 크기 문제가 있어 자동으로 해상도/프레임을 줄이지만, 그래도 mp4보다
# 파일이 커질 수 있다 (포맷 자체의 압축 한계).
OVERLAY_FORMAT = "mp4"

overlay_paths = {}
for label, paths in DEMO_VIDEOS.items():
    video_path = Path(paths["video_path"])
    if not video_path.exists():
        print(f"[건너뜀] {label}: {video_path} 없음 — 경로를 실제 시연 영상으로 바꿔주세요.")
        continue

    output_path = video_path.with_name(video_path.stem + "_overlay")
    output_path = overlay_skeleton_on_video(
        video_path=video_path,
        keypoints_path=paths["keypoints_path"],
        output_path=output_path,
        output_format=OVERLAY_FORMAT,
    )
    overlay_paths[label] = output_path

# 발표 중에는 이 중 하나를 골라 재생하면서 시작하면 좋습니다.
if overlay_paths:
    first_label = next(iter(overlay_paths))
    print(f"\n▶ 미리보기: {first_label}")
    if OVERLAY_FORMAT == "mp4":
        display(Video(str(overlay_paths[first_label]), embed=True, width=360))
    else:
        from IPython.display import Image
        display(Image(str(overlay_paths[first_label])))


## 3. 전처리 전/후 스켈레톤 비교

`coordiante_normalization.normalize_landmarks()`가 실제로 뭘 하는지 눈으로 보여줍니다.
`animate_skeleton_2d()`는 항상 원점(0,0)을 금색 삼각형으로 표시하는데:
- **전처리 전(원본 좌표)**: 원점이 몸에서 멀리 떨어진 화면 구석에 찍힙니다 (이미지 좌표계
  원점이라 몸의 위치와 무관합니다)
- **전처리 후(힙 중심 정렬 + 스케일 정규화)**: 원점이 골반 중앙, 즉 몸의 중심에 정확히 찍힙니다

같은 좌표 시퀀스인데 기준점만 바뀌었다는 걸 그대로 눈으로 확인할 수 있습니다
(`Test.ipynb`에서 쓰던 것과 같은 `animate_skeleton_2d`를 재사용합니다).

In [ ]:
from src.preprocessing.build_dataset import _mask_low_visibility, _interpolate_missing_frames
from src.preprocessing.coordiante_normalization import normalize_landmarks
from src.preprocessing.visualizer import animate_skeleton_2d
import re

# DEMO_VIDEOS 중 존재하는 첫 번째 영상의 원본 keypoints로 비교
compare_label = next((l for l in DEMO_VIDEOS if Path(DEMO_VIDEOS[l]["keypoints_path"]).exists()), None)

if compare_label is None:
    print("비교할 keypoints가 없습니다 — DEMO_VIDEOS 경로를 확인하세요.")
else:
    raw_keypoints = np.load(DEMO_VIDEOS[compare_label]["keypoints_path"])

    # normalize_landmarks는 저신뢰/미검출 프레임(NaN)이 섞여 있으면 정규화 도중
    # NaN/Inf가 전파될 수 있다 (실측 확인됨). build_dataset과 동일하게 마스킹 ->
    # 보간을 먼저 거쳐야 "전처리 후" 비교가 안전하다.
    masked = _mask_low_visibility(raw_keypoints, threshold=0.5)
    if np.isnan(masked).any():
        masked = _interpolate_missing_frames(masked, fallback_raw=raw_keypoints)
    normalized_keypoints = normalize_landmarks(masked, n_spatial_dims=2)

    # 라벨에 괄호/한글/공백이 섞여 있어도 안전한 파일명이 되도록 영숫자 위주로 슬러그화
    slug = re.sub(r"[^0-9a-zA-Z]+", "_", compare_label).strip("_")[:30] or "compare"
    before_path = f"tests/{slug}_before_normalize.gif"
    after_path = f"tests/{slug}_after_normalize.gif"

    print(f"'{compare_label}' 기준으로 비교 (원본 프레임 {raw_keypoints.shape[0]}개)")
    animate_skeleton_2d(raw_keypoints, save_path=before_path, title="전처리 전 (원본 좌표)")
    animate_skeleton_2d(normalized_keypoints, save_path=after_path, title="전처리 후 (힙 중심 정렬)")

    display(Image(before_path))
    display(Image(after_path))


## 4. 세 영상 모두 채점 (통계 + 규칙 기반)

`score_video()` 하나로 전처리(DTW 위상정규화 포함) + 채점까지 전부 처리됩니다.
`score_source` 컬럼으로 최종 점수가 통계(mahalanobis)와 규칙(rule) 중 어디서
나왔는지 바로 확인할 수 있습니다.

In [ ]:
all_results = {}
for label, paths in DEMO_VIDEOS.items():
    keypoints_path = Path(paths["keypoints_path"])
    if not keypoints_path.exists():
        continue
    results = score_video(
        keypoints_path=keypoints_path,
        exercise=EXERCISE,
        template=template,
        model_dir=MODEL_DIR,
    )
    all_results[label] = results

summary_rows = []
for label, results in all_results.items():
    for r in results:
        summary_rows.append({
            "영상": label, "rep_idx": r["rep_idx"], "score": round(r["score"], 1),
            "score_source": r["score_source"], "distance": round(r["distance"], 2),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df

## 5. 핵심 시연 포인트 — 규칙 기반 최저점 보장 확인

세 번째 영상이 **mahalanobis 점수만 썼다면 몇 점이었을지**와, **규칙까지 합친
실제 최종 점수**를 비교합니다. 규칙은 "잘하면 더 준다"가 아니라 **"최소 동작
기준(게이트)만 통과하면 최대 50점까지는 보장한다"는 안전망**이라, mahalanobis가
이미 규칙 점수보다 높다면 규칙은 아무 영향도 주지 않습니다 — 좋은 폼/나쁜 폼
영상의 점수가 규칙과 무관하게 원래 통계 점수 그대로인 것도 같이 확인합니다.

In [ ]:
outlier_label = "통계상 큰 이상치지만 최소 동작은 충족 (규칙 최저점 보장 시연)"
if outlier_label in all_results:
    for r in all_results[outlier_label]:
        print(f"rep_idx={r['rep_idx']}")
        print(f"  최종 점수: {r['score']:.1f}점 (출처: {r['score_source']})")
        if r["score_source"] == "rule":
            print(f"  -> mahalanobis 점수만 썼다면 이 rep은 규칙 최저점({r['rule_explanation']['score']:.1f}점)보다")
            print(f"     낮았을 것입니다 (최소 동작 기준은 통과했으므로 여기까지는 보장됩니다).")
            print(f"  -> 규칙 계산 과정:")
            for m in r["rule_explanation"]["messages"]:
                print(f"       {m}")
        else:
            print(f"  -> 이 rep은 mahalanobis 점수가 이미 규칙 최저점보다 높아서, 규칙이 관여하지 않았습니다.")
        print()
else:
    print(f"'{outlier_label}' 영상이 준비 안 됐습니다 — DEMO_VIDEOS 경로를 확인하세요.")

## 6. rep별 이상치 리포트 (언어화)

`outlier_messages`는 특징 이름(`knee_bin03` 등)을 사람이 읽을 수 있는 문장으로
바꾼 것입니다. phase-bin 특징은 "rep 진행 30~40% 구간"처럼 **몇 % 지점에서
문제가 생겼는지**까지 그대로 드러납니다.

In [ ]:
bad_label = "나쁜 폼"
if bad_label in all_results:
    for r in all_results[bad_label]:
        print(f"[{bad_label}] rep_idx={r['rep_idx']} score={r['score']:.1f} (출처: {r['score_source']})")
        if r["outlier_messages"]:
            print("  발견된 이상치:")
            for msg in r["outlier_messages"]:
                print(f"   - {msg}")
        else:
            print("  이상치 없음 (기준 범위 안)")
        print()
else:
    print(f"'{bad_label}' 영상이 준비 안 됐습니다 — DEMO_VIDEOS 경로를 확인하세요.")

## 7. 세 영상 점수 한눈에 비교 (시각화)

In [ ]:
if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = {"mahalanobis": "tab:blue", "rule": "tab:orange"}
    for label in summary_df["영상"].unique():
        sub = summary_df[summary_df["영상"] == label]
        for _, row in sub.iterrows():
            ax.bar(f"{label}\n(rep {row['rep_idx']})", row["score"], color=colors.get(row["score_source"], "gray"))

    ax.set_ylabel("score"); ax.set_ylim(0, 100)
    ax.set_title("영상별 최종 점수 (파랑=통계 기준, 주황=규칙 기반 보정)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("채점된 결과가 없습니다 — DEMO_VIDEOS 경로를 먼저 채워주세요.")